<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_4_%E2%80%94_CROPLAND_LOSS_AND_AGRICULTURAL_CONVERSION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
```python
# =============================================================================
# TZPR–NLP NATIONAL HIGHWAY CORRIDOR
# SECTION 4.4 — CROPLAND LOSS AND AGRICULTURAL CONVERSION
# =============================================================================
#
# PURPOSE
# -------
# Extract manuscript-ready statistics from exported GeoTIFF files for:
#
#   1. Cropland area in 2016
#   2. Cropland area in 2025
#   3. Net cropland loss
#   4. Percentage cropland loss
#   5. Average annual cropland loss
#   6. Directly mapped cropland-loss area
#   7. Cropland → Built-up conversion
#   8. Cropland → Bare conversion
#   9. Conversion contribution percentages
#  10. Raster consistency checks
#  11. Manuscript-ready text summary
#
# EXPECTED INPUT FILES
# --------------------
# TZPR_NLP_Cropland_2016.tif
# TZPR_NLP_Cropland_2025.tif
# TZPR_NLP_Cropland_Loss_2016_2025.tif
# TZPR_NLP_Crops_to_Built_2016_2025.tif
# TZPR_NLP_Crops_to_Bare_2016_2025.tif
#
# OUTPUT DIRECTORY
# ----------------
# Cropland_Analysis_4_4/
#
# =============================================================================


import os
import glob
import warnings

import numpy as np
import pandas as pd
import rasterio


warnings.filterwarnings("ignore")


# =============================================================================
# 1. USER SETTINGS
# =============================================================================

# -------------------------------------------------------------------------
# CHANGE THIS PATH IF REQUIRED
# -------------------------------------------------------------------------

INPUT_DIR = "/content/drive/MyDrive/TZPR_NLP_Research"

# Output folder
OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Cropland_Analysis_4_4"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Study years
START_YEAR = 2016
END_YEAR = 2025

# Number of years between endpoint observations
YEAR_INTERVAL = END_YEAR - START_YEAR


# =============================================================================
# 2. EXPECTED FILES
# =============================================================================

FILES = {

    "cropland_2016":
        "TZPR_NLP_Cropland_2016.tif",

    "cropland_2025":
        "TZPR_NLP_Cropland_2025.tif",

    "cropland_loss":
        "TZPR_NLP_Cropland_Loss_2016_2025.tif",

    "crops_to_built":
        "TZPR_NLP_Crops_to_Built_2016_2025.tif",

    "crops_to_bare":
        "TZPR_NLP_Crops_to_Bare_2016_2025.tif",

}


# =============================================================================
# 3. AREA CONVERSION FUNCTIONS
# =============================================================================

HECTARE_TO_KM2 = 0.01
M2_TO_HECTARE = 0.0001


def format_number(value, decimals=2):
    """
    Format numbers with commas.
    """
    if value is None or np.isnan(value):
        return "NA"

    return f"{value:,.{decimals}f}"


def area_from_binary_raster(filepath):
    """
    Calculate area represented by valid/non-zero pixels.

    Assumes:
        - Cropland/loss/transition rasters are binary masks
        - valid positive pixels represent the target class
        - nodata pixels are excluded
    """

    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"File not found:\n{filepath}"
        )

    with rasterio.open(filepath) as src:

        data = src.read(1, masked=True)

        # Pixel dimensions
        pixel_width = abs(src.transform.a)
        pixel_height = abs(src.transform.e)

        pixel_area_m2 = pixel_width * pixel_height

        # Convert masked array to ndarray
        values = data.filled(np.nan)

        # Target pixels
        target = (
            np.isfinite(values) &
            (values > 0)
        )

        pixel_count = int(np.sum(target))

        area_m2 = pixel_count * pixel_area_m2
        area_ha = area_m2 * M2_TO_HECTARE
        area_km2 = area_ha * HECTARE_TO_KM2

        return {
            "file": os.path.basename(filepath),
            "pixel_count": pixel_count,
            "pixel_width_m": pixel_width,
            "pixel_height_m": pixel_height,
            "pixel_area_m2": pixel_area_m2,
            "area_m2": area_m2,
            "area_ha": area_ha,
            "area_km2": area_km2,
            "crs": str(src.crs),
            "nodata": src.nodata,
            "width": src.width,
            "height": src.height,
        }


# =============================================================================
# 4. RASTER INFORMATION CHECK
# =============================================================================

def raster_info(filepath):

    if not os.path.exists(filepath):
        return {
            "file": os.path.basename(filepath),
            "exists": False
        }

    with rasterio.open(filepath) as src:

        data = src.read(1, masked=True)

        valid = data.compressed()

        if len(valid) > 0:

            min_val = float(np.min(valid))
            max_val = float(np.max(valid))
            mean_val = float(np.mean(valid))

            unique_sample = np.unique(valid)

            # Avoid extremely large unique arrays
            if len(unique_sample) <= 20:
                unique_values = ",".join(
                    [str(x) for x in unique_sample]
                )
            else:
                unique_values = f"{len(unique_sample)} unique values"

        else:

            min_val = np.nan
            max_val = np.nan
            mean_val = np.nan
            unique_values = "No valid pixels"

        return {
            "file": os.path.basename(filepath),
            "exists": True,
            "width": src.width,
            "height": src.height,
            "crs": str(src.crs),
            "pixel_width_m": abs(src.transform.a),
            "pixel_height_m": abs(src.transform.e),
            "nodata": src.nodata,
            "valid_pixels": int(np.sum(~data.mask)),
            "min": min_val,
            "max": max_val,
            "mean": mean_val,
            "unique_values": unique_values
        }


# =============================================================================
# 5. LOCATE INPUT FILES
# =============================================================================

print("\n")
print("=" * 80)
print("TZPR–NLP SECTION 4.4")
print("CROPLAND LOSS AND AGRICULTURAL CONVERSION")
print("=" * 80)

print("\nInput directory:")
print(INPUT_DIR)

print("\nChecking input files...\n")


FILE_PATHS = {}

missing_files = []

for key, filename in FILES.items():

    filepath = os.path.join(INPUT_DIR, filename)

    FILE_PATHS[key] = filepath

    if os.path.exists(filepath):

        print(f"[OK]      {filename}")

    else:

        print(f"[MISSING] {filename}")
        missing_files.append(filename)


if missing_files:

    print("\n" + "=" * 80)
    print("WARNING: SOME REQUIRED FILES ARE MISSING")
    print("=" * 80)

    for f in missing_files:
        print(" -", f)

    print(
        "\nPlease check INPUT_DIR and make sure the exported "
        "GeoTIFF files are present."
    )

    raise FileNotFoundError(
        "Required GeoTIFF files are missing."
    )


# =============================================================================
# 6. RASTER QUALITY / METADATA REPORT
# =============================================================================

print("\n")
print("=" * 80)
print("RASTER INFORMATION")
print("=" * 80)

raster_info_records = []

for key, filepath in FILE_PATHS.items():

    info = raster_info(filepath)

    info["dataset"] = key

    raster_info_records.append(info)

    print("\nDataset:", key)
    print("File:", info["file"])
    print("Dimensions:", info["width"], "x", info["height"])
    print(
        "Pixel size:",
        info["pixel_width_m"],
        "x",
        info["pixel_height_m"],
        "m"
    )
    print("CRS:", info["crs"])
    print("NoData:", info["nodata"])
    print("Valid pixels:", info["valid_pixels"])
    print("Minimum:", info["min"])
    print("Maximum:", info["max"])
    print("Mean:", info["mean"])
    print("Unique values:", info["unique_values"])


raster_info_df = pd.DataFrame(raster_info_records)

raster_info_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "01_Cropland_Raster_Information.csv"
    ),
    index=False
)


# =============================================================================
# 7. CALCULATE CROPLAND AREAS
# =============================================================================

print("\n")
print("=" * 80)
print("CROPLAND AREA CALCULATION")
print("=" * 80)


area_results = {}

for key in [
    "cropland_2016",
    "cropland_2025",
    "cropland_loss",
    "crops_to_built",
    "crops_to_bare"
]:

    result = area_from_binary_raster(
        FILE_PATHS[key]
    )

    area_results[key] = result

    print(
        f"\n{key}: "
        f"{result['area_ha']:,.2f} ha "
        f"({result['area_km2']:,.2f} km²)"
    )


# =============================================================================
# 8. MAIN CROPLAND CHANGE STATISTICS
# =============================================================================

cropland_2016_ha = (
    area_results["cropland_2016"]["area_ha"]
)

cropland_2025_ha = (
    area_results["cropland_2025"]["area_ha"]
)

cropland_loss_mapped_ha = (
    area_results["cropland_loss"]["area_ha"]
)

cropland_2016_km2 = cropland_2016_ha * HECTARE_TO_KM2
cropland_2025_km2 = cropland_2025_ha * HECTARE_TO_KM2
cropland_loss_mapped_km2 = (
    cropland_loss_mapped_ha * HECTARE_TO_KM2
)


# Endpoint net change
net_change_ha = (
    cropland_2025_ha -
    cropland_2016_ha
)

# Positive value represents loss
net_loss_ha = abs(net_change_ha)

net_loss_km2 = net_loss_ha * HECTARE_TO_KM2


# Percentage loss relative to 2016
if cropland_2016_ha > 0:

    percentage_loss = (
        net_loss_ha /
        cropland_2016_ha
    ) * 100

else:

    percentage_loss = np.nan


# Remaining proportion
if cropland_2016_ha > 0:

    remaining_percentage = (
        cropland_2025_ha /
        cropland_2016_ha
    ) * 100

else:

    remaining_percentage = np.nan


# Average annual loss
if YEAR_INTERVAL > 0:

    annual_loss_ha = (
        net_loss_ha /
        YEAR_INTERVAL
    )

else:

    annual_loss_ha = np.nan


annual_loss_km2 = (
    annual_loss_ha *
    HECTARE_TO_KM2
)


# =============================================================================
# 9. CROPLAND CONVERSION PATHWAYS
# =============================================================================

crops_to_built_ha = (
    area_results["crops_to_built"]["area_ha"]
)

crops_to_built_km2 = (
    crops_to_built_ha *
    HECTARE_TO_KM2
)


crops_to_bare_ha = (
    area_results["crops_to_bare"]["area_ha"]
)

crops_to_bare_km2 = (
    crops_to_bare_ha *
    HECTARE_TO_KM2
)


# Combined mapped conversion
combined_conversion_ha = (
    crops_to_built_ha +
    crops_to_bare_ha
)

combined_conversion_km2 = (
    combined_conversion_ha *
    HECTARE_TO_KM2
)


# =============================================================================
# 10. CONVERSION CONTRIBUTION
# =============================================================================

if combined_conversion_ha > 0:

    built_conversion_percentage = (
        crops_to_built_ha /
        combined_conversion_ha
    ) * 100

    bare_conversion_percentage = (
        crops_to_bare_ha /
        combined_conversion_ha
    ) * 100

else:

    built_conversion_percentage = np.nan
    bare_conversion_percentage = np.nan


# Contribution relative to total mapped cropland loss
if cropland_loss_mapped_ha > 0:

    built_of_loss_percentage = (
        crops_to_built_ha /
        cropland_loss_mapped_ha
    ) * 100

    bare_of_loss_percentage = (
        crops_to_bare_ha /
        cropland_loss_mapped_ha
    ) * 100

else:

    built_of_loss_percentage = np.nan
    bare_of_loss_percentage = np.nan


# =============================================================================
# 11. CONSISTENCY CHECK
# =============================================================================

difference_ha = (
    cropland_loss_mapped_ha -
    net_loss_ha
)

difference_percentage = (
    abs(difference_ha) /
    net_loss_ha *
    100
    if net_loss_ha > 0
    else np.nan
)


# =============================================================================
# 12. PRINT MAIN RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("MAIN CROPLAND RESULTS")
print("=" * 80)

print(
    f"\nCropland area in {START_YEAR}: "
    f"{cropland_2016_ha:,.2f} ha "
    f"({cropland_2016_km2:,.2f} km²)"
)

print(
    f"Cropland area in {END_YEAR}: "
    f"{cropland_2025_ha:,.2f} ha "
    f"({cropland_2025_km2:,.2f} km²)"
)

print(
    f"\nNet cropland loss: "
    f"{net_loss_ha:,.2f} ha "
    f"({net_loss_km2:,.2f} km²)"
)

print(
    f"Percentage loss: "
    f"{percentage_loss:,.2f}%"
)

print(
    f"Average annual loss: "
    f"{annual_loss_ha:,.2f} ha/year "
    f"({annual_loss_km2:,.2f} km²/year)"
)

print(
    f"\nRemaining cropland in {END_YEAR}: "
    f"{remaining_percentage:,.2f}% "
    f"of {START_YEAR} cropland"
)


# =============================================================================
# 13. PRINT CONVERSION RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("CROPLAND CONVERSION PATHWAYS")
print("=" * 80)

print(
    f"\nCropland → Built-up: "
    f"{crops_to_built_ha:,.2f} ha "
    f"({crops_to_built_km2:,.2f} km²)"
)

print(
    f"Cropland → Bare: "
    f"{crops_to_bare_ha:,.2f} ha "
    f"({crops_to_bare_km2:,.2f} km²)"
)

print(
    f"\nCombined mapped conversion: "
    f"{combined_conversion_ha:,.2f} ha "
    f"({combined_conversion_km2:,.2f} km²)"
)

print(
    f"\nBuilt-up share of mapped conversion: "
    f"{built_conversion_percentage:,.2f}%"
)

print(
    f"Bare-land share of mapped conversion: "
    f"{bare_conversion_percentage:,.2f}%"
)


# =============================================================================
# 14. LOSS CONSISTENCY
# =============================================================================

print("\n")
print("=" * 80)
print("CROPLAND LOSS CONSISTENCY CHECK")
print("=" * 80)

print(
    f"\nEndpoint net loss: "
    f"{net_loss_ha:,.2f} ha"
)

print(
    f"Mapped cropland-loss raster: "
    f"{cropland_loss_mapped_ha:,.2f} ha"
)

print(
    f"Difference: "
    f"{difference_ha:,.2f} ha"
)

print(
    f"Relative difference: "
    f"{difference_percentage:,.2f}%"
)


if abs(difference_percentage) < 5:

    print(
        "\nSTATUS: GOOD AGREEMENT between endpoint "
        "change and mapped cropland-loss raster."
    )

else:

    print(
        "\nSTATUS: NOTICEABLE DIFFERENCE detected. "
        "Check raster definitions, class masks, or "
        "classification-year comparison."
    )


# =============================================================================
# 15. MANUSCRIPT SUMMARY TABLE
# =============================================================================

summary_records = [

    {
        "Metric": f"Cropland area {START_YEAR}",
        "Area_ha": cropland_2016_ha,
        "Area_km2": cropland_2016_km2,
        "Percentage": 100.0
    },

    {
        "Metric": f"Cropland area {END_YEAR}",
        "Area_ha": cropland_2025_ha,
        "Area_km2": cropland_2025_km2,
        "Percentage": remaining_percentage
    },

    {
        "Metric": "Net cropland loss",
        "Area_ha": net_loss_ha,
        "Area_km2": net_loss_km2,
        "Percentage": percentage_loss
    },

    {
        "Metric": "Mapped cropland-loss area",
        "Area_ha": cropland_loss_mapped_ha,
        "Area_km2": cropland_loss_mapped_km2,
        "Percentage": (
            cropland_loss_mapped_ha /
            cropland_2016_ha * 100
            if cropland_2016_ha > 0
            else np.nan
        )
    },

    {
        "Metric": "Average annual cropland loss",
        "Area_ha": annual_loss_ha,
        "Area_km2": annual_loss_km2,
        "Percentage": np.nan
    },

    {
        "Metric": "Cropland → Built-up",
        "Area_ha": crops_to_built_ha,
        "Area_km2": crops_to_built_km2,
        "Percentage": built_of_loss_percentage
    },

    {
        "Metric": "Cropland → Bare",
        "Area_ha": crops_to_bare_ha,
        "Area_km2": crops_to_bare_km2,
        "Percentage": bare_of_loss_percentage
    },

    {
        "Metric": "Combined mapped conversion",
        "Area_ha": combined_conversion_ha,
        "Area_km2": combined_conversion_km2,
        "Percentage": (
            combined_conversion_ha /
            cropland_loss_mapped_ha * 100
            if cropland_loss_mapped_ha > 0
            else np.nan
        )
    }

]


summary_df = pd.DataFrame(summary_records)


summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "02_Cropland_Manuscript_Summary.csv"
    ),
    index=False
)


# =============================================================================
# 16. DETAILED CHANGE STATISTICS
# =============================================================================

change_records = {

    "Start_Year": START_YEAR,

    "End_Year": END_YEAR,

    "Year_Interval": YEAR_INTERVAL,

    "Cropland_2016_ha": cropland_2016_ha,

    "Cropland_2016_km2": cropland_2016_km2,

    "Cropland_2025_ha": cropland_2025_ha,

    "Cropland_2025_km2": cropland_2025_km2,

    "Net_Cropland_Loss_ha": net_loss_ha,

    "Net_Cropland_Loss_km2": net_loss_km2,

    "Cropland_Loss_Percent": percentage_loss,

    "Remaining_Cropland_Percent": remaining_percentage,

    "Average_Annual_Loss_ha": annual_loss_ha,

    "Average_Annual_Loss_km2": annual_loss_km2,

    "Mapped_Cropland_Loss_ha": cropland_loss_mapped_ha,

    "Mapped_Cropland_Loss_km2": cropland_loss_mapped_km2,

    "Cropland_to_Built_ha": crops_to_built_ha,

    "Cropland_to_Built_km2": crops_to_built_km2,

    "Cropland_to_Bare_ha": crops_to_bare_ha,

    "Cropland_to_Bare_km2": crops_to_bare_km2,

    "Combined_Conversion_ha": combined_conversion_ha,

    "Combined_Conversion_km2": combined_conversion_km2,

    "Built_Conversion_Share_Percent":
        built_conversion_percentage,

    "Bare_Conversion_Share_Percent":
        bare_conversion_percentage,

    "Built_as_Percent_of_Mapped_Cropland_Loss":
        built_of_loss_percentage,

    "Bare_as_Percent_of_Mapped_Cropland_Loss":
        bare_of_loss_percentage,

    "Loss_Raster_vs_Endpoint_Difference_ha":
        difference_ha,

    "Loss_Raster_vs_Endpoint_Difference_Percent":
        difference_percentage
}


change_df = pd.DataFrame(
    [change_records]
)


change_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "03_Cropland_Change_Statistics.csv"
    ),
    index=False
)


# =============================================================================
# 17. CONVERSION TABLE
# =============================================================================

conversion_df = pd.DataFrame({

    "Conversion_Pathway": [
        "Cropland → Built-up",
        "Cropland → Bare"
    ],

    "Area_ha": [
        crops_to_built_ha,
        crops_to_bare_ha
    ],

    "Area_km2": [
        crops_to_built_km2,
        crops_to_bare_km2
    ],

    "Percent_of_Combined_Mapped_Conversion": [
        built_conversion_percentage,
        bare_conversion_percentage
    ],

    "Percent_of_Mapped_Cropland_Loss": [
        built_of_loss_percentage,
        bare_of_loss_percentage
    ]

})


conversion_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "04_Cropland_Conversion_Pathways.csv"
    ),
    index=False
)


# =============================================================================
# 18. RASTER AREA CHECK TABLE
# =============================================================================

raster_area_df = pd.DataFrame({

    "Dataset": [
        "Cropland 2016",
        "Cropland 2025",
        "Cropland Loss 2016-2025",
        "Crops to Built 2016-2025",
        "Crops to Bare 2016-2025"
    ],

    "GeoTIFF": [
        FILES["cropland_2016"],
        FILES["cropland_2025"],
        FILES["cropland_loss"],
        FILES["crops_to_built"],
        FILES["crops_to_bare"]
    ],

    "Area_ha": [
        cropland_2016_ha,
        cropland_2025_ha,
        cropland_loss_mapped_ha,
        crops_to_built_ha,
        crops_to_bare_ha
    ],

    "Area_km2": [
        cropland_2016_km2,
        cropland_2025_km2,
        cropland_loss_mapped_km2,
        crops_to_built_km2,
        crops_to_bare_km2
    ],

    "Pixel_Count": [
        area_results["cropland_2016"]["pixel_count"],
        area_results["cropland_2025"]["pixel_count"],
        area_results["cropland_loss"]["pixel_count"],
        area_results["crops_to_built"]["pixel_count"],
        area_results["crops_to_bare"]["pixel_count"]
    ]

})


raster_area_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05_Cropland_Raster_Area_Check.csv"
    ),
    index=False
)


# =============================================================================
# 19. MANUSCRIPT-READY TEXT FILE
# =============================================================================

manuscript_text = f"""
===============================================================================
SECTION 4.4 — CROPLAND LOSS AND AGRICULTURAL CONVERSION
===============================================================================

Study period:
{START_YEAR}–{END_YEAR}

CROPLAND AREA
-------------

Cropland area in {START_YEAR}:
{cropland_2016_ha:,.2f} ha
({cropland_2016_km2:,.2f} km²)

Cropland area in {END_YEAR}:
{cropland_2025_ha:,.2f} ha
({cropland_2025_km2:,.2f} km²)

NET CROPLAND CHANGE
-------------------

Net cropland loss:
{net_loss_ha:,.2f} ha
({net_loss_km2:,.2f} km²)

Percentage cropland loss:
{percentage_loss:,.2f}%

Average annual cropland loss:
{annual_loss_ha:,.2f} ha/year
({annual_loss_km2:,.2f} km²/year)

Remaining cropland in {END_YEAR}:
{remaining_percentage:,.2f}% of the {START_YEAR} cropland area


MAPPED CROPLAND LOSS
--------------------

Cropland-loss raster:
{cropland_loss_mapped_ha:,.2f} ha
({cropland_loss_mapped_km2:,.2f} km²)

Difference from endpoint-derived loss:
{difference_ha:,.2f} ha

Relative difference:
{difference_percentage:,.2f}%


CROPLAND CONVERSION PATHWAYS
----------------------------

Cropland → Built-up:
{crops_to_built_ha:,.2f} ha
({crops_to_built_km2:,.2f} km²)

Cropland → Bare:
{crops_to_bare_ha:,.2f} ha
({crops_to_bare_km2:,.2f} km²)

Combined mapped conversion:
{combined_conversion_ha:,.2f} ha
({combined_conversion_km2:,.2f} km²)

Built-up share of combined mapped conversion:
{built_conversion_percentage:,.2f}%

Bare-land share of combined mapped conversion:
{bare_conversion_percentage:,.2f}%


INTERPRETATION NOTES
--------------------

1. The endpoint-derived cropland loss is calculated as the difference between
   cropland area in {START_YEAR} and {END_YEAR}.

2. The cropland-loss GeoTIFF represents the pixels mapped as cropland loss
   during the study period.

3. Cropland → Built-up and Cropland → Bare are specific conversion pathways.

4. The mapped cropland-loss area should not automatically be assumed to equal
   the sum of individual conversion pathways unless the transition layers
   collectively represent all cropland-loss destinations.

5. If the cropland-loss raster and endpoint-derived loss differ, report them
   separately rather than forcing them to match.

6. Cropland → Built-up should be interpreted as a specific land-cover
   conversion pathway and not automatically as the sole driver of cropland
   loss.

7. The spatial pattern shown in Figure 6 should be used to describe where
   cropland loss occurs. Avoid claiming concentration near specific towns,
   settlements, junctions, or road segments unless this is clearly visible
   and independently supported by the spatial analysis.


FIGURE 6
--------

Spatial distribution of cropland and cropland loss between {START_YEAR}
and {END_YEAR}.

Panels:
(a) TZPR_NLP_Cropland_2016
(b) TZPR_NLP_Cropland_2025
(c) TZPR_NLP_Cropland_Loss_2016_2025


FIGURE 7
--------

Major cropland conversion pathways during {START_YEAR}–{END_YEAR}.

Panels:
(a) TZPR_NLP_Crops_to_Built_2016_2025
(b) TZPR_NLP_Crops_to_Bare_2016_2025

===============================================================================
"""


text_output = os.path.join(
    OUTPUT_DIR,
    "06_Cropland_Manuscript_Values.txt"
)


with open(
    text_output,
    "w",
    encoding="utf-8"
) as f:

    f.write(manuscript_text)


# =============================================================================
# 20. FINAL OUTPUT SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("SECTION 4.4 ANALYSIS COMPLETED")
print("=" * 80)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

output_files = sorted(
    glob.glob(
        os.path.join(
            OUTPUT_DIR,
            "*"
        )
    )
)

for file in output_files:

    print(
        " -",
        os.path.basename(file)
    )


print("\n")
print("=" * 80)
print("KEY MANUSCRIPT VALUES")
print("=" * 80)

print(
    f"\nCropland {START_YEAR}: "
    f"{cropland_2016_ha:,.2f} ha"
)

print(
    f"Cropland {END_YEAR}: "
    f"{cropland_2025_ha:,.2f} ha"
)

print(
    f"Net loss: "
    f"{net_loss_ha:,.2f} ha "
    f"({percentage_loss:,.2f}%)"
)

print(
    f"Annual loss: "
    f"{annual_loss_ha:,.2f} ha/year"
)

print(
    f"Cropland → Built-up: "
    f"{crops_to_built_ha:,.2f} ha"
)

print(
    f"Cropland → Bare: "
    f"{crops_to_bare_ha:,.2f} ha"
)

print("\n")
print("Analysis complete.")
print("=" * 80)
```
